<a href="https://colab.research.google.com/github/BassemRamdan/AI-Resume-Intelligence/blob/main/notebooks/04_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 4: NLP Pipeline and Weak Labeling Exploration

**Goal:**
- Reconstruct clean full text from the Layout JSON created in Phase 3.
- Apply NLP cleaning and normalization using `spaCy`.
- Investigate heuristic rules to generate "Weak Labels" for Resume Sections (e.g., EDUCATION, EXPERIENCE), paving the way for LayoutLMv3 in Phase 6.

In [ ]:
!pip install spacy pandas tqdm huggingface_hub
!python -m spacy download en_core_web_sm

In [ ]:
import os
import json
import re
import pandas as pd
import spacy
from tqdm.auto import tqdm

nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"]) # Disable heavy pipelines we don't need yet

# 1. Load the Layout JSON from Phase 3
# Note: If you pushed it to HF, you can download it via hf_hub_download. 
# For now, we assume it's available locally or uploaded to Colab.
layout_path = "resume_layout_dataset.json"

if not os.path.exists(layout_path):
    print("Please ensure 'resume_layout_dataset.json' from Phase 3 is uploaded to your Colab environment.")
else:
    with open(layout_path, 'r', encoding='utf-8') as f:
        layout_data = json.load(f)
    print(f"Loaded {len(layout_data)} structured resumes.")

In [ ]:
def reconstruct_and_clean_text(pages):
    """Reconstructs full text from bounding box words and applies basic regex cleaning."""
    full_text = []
    for page in pages:
        for word_obj in page['words']:
            full_text.append(word_obj['text'])
            
    raw_text = " ".join(full_text)
    # Remove excessive whitespaces and special characters but keep basic punctuation
    clean_text = re.sub(r'\s+', ' ', raw_text)
    clean_text = re.sub(r'[^\w\s.,;:\-@/]', '', clean_text)
    return clean_text.strip()

def lemmatize_text(text):
    """Lemmatizes text and removes stop words using spaCy."""
    doc = nlp(text)
    tokens = [token.lemma_.lower() for token in doc if not token.is_stop and not token.is_punct and token.is_alpha]
    return " ".join(tokens)

print("Processing NLP Pipeline (Reconstruction -> Regex Cleaning -> Lemmatization)...")
processed_data = []

for resume in tqdm(layout_data, desc="NLP Pipeline"):
    raw_clean_text = reconstruct_and_clean_text(resume['pages'])
    lemmatized_text = lemmatize_text(raw_clean_text)
    
    processed_data.append({
        "filename": resume['filename'],
        "category": resume['category'],
        "raw_clean_text": raw_clean_text,
        "lemmatized_text": lemmatized_text
    })

nlp_df = pd.DataFrame(processed_data)
nlp_df.to_csv("nlp_processed_resumes.csv", index=False)
print(f"\nSaved processed text to 'nlp_processed_resumes.csv'. Shape: {nlp_df.shape}")

In [ ]:
print("--- Weak Labeling Exploration for LayoutLMv3 ---")
# We explore heuristics to detect section headers. 
# For example: capitalized words matching known sections (EDUCATION, SKILLS, EXPERIENCE)

known_sections = ["EDUCATION", "EXPERIENCE", "SKILLS", "SUMMARY", "PROJECTS", "CERTIFICATIONS", "LANGUAGES"]
section_pattern = re.compile(r'\b(' + '|'.join(known_sections) + r')\b', re.IGNORECASE)

# Let's check how many resumes contain these keywords as potential headers
section_counts = {sec: 0 for sec in known_sections}

for text in nlp_df['raw_clean_text']:
    found = set(re.findall(section_pattern, text))
    for f in found:
        section_counts[f.upper()] += 1

print("Percentage of Resumes containing specific section keywords (Potential for weak labeling):")
for sec, count in section_counts.items():
    pct = (count / len(nlp_df)) * 100
    print(f"{sec}: {pct:.2f}%")

print("\nConclusion: Many resumes contain these keywords. In Phase 6, we can use font-size heuristics combined with these keywords to automatically generate LayoutLMv3 Bounding Box labels!")